In [1]:
import pandas as pd
import csv
import requests
import zipfile
from pathlib import Path

In [2]:
url = "https://nemweb.com.au/Reports/CURRENT/DispatchIS_Reports/PUBLIC_DISPATCHIS_202611150515_0000000548649038.zip"
url_trimmed = url.removeprefix("https://nemweb.com.au/Reports/CURRENT/DispatchIS_Reports/")

path = Path(f"../data/raw/{url_trimmed}")
if not path.exists():
    file = requests.get(url, timeout=30)
    file.raise_for_status()
    
    with open(f"../data/raw/{url_trimmed}", "xb") as f:
        f.write(file.content)

In [3]:
with zipfile.ZipFile(f"../data/raw/{url_trimmed}", "r") as zip_f:
    zip_f.extractall(f"../data/tmp/")

In [4]:
try:
    pd.read_csv("../data/tmp/PUBLIC_DISPATCHIS_202611150515_0000000548649038.CSV")
except pd.errors.ParserError as e:
    print(e)

Error tokenizing data. C error: Expected 28 fields in line 7, saw 70



The file contains multiple tables that stacks on top of each other but `read_csv` can only handle one. \
28 fields are from the first table in the file (`CASE_SOLUTION`), not from `PRICE`.

In [5]:
price_table = []

with open("../data/tmp/PUBLIC_DISPATCHIS_202611150515_0000000548649038.CSV", "r") as f:
    for row in csv.reader(f):
        if row[1] == "DISPATCH" and row[2] == "PRICE":
            price_table.append(row)

print(price_table)

[['I', 'DISPATCH', 'PRICE', '5', 'SETTLEMENTDATE', 'RUNNO', 'REGIONID', 'DISPATCHINTERVAL', 'INTERVENTION', 'RRP', 'EEP', 'ROP', 'APCFLAG', 'MARKETSUSPENDEDFLAG', 'LASTCHANGED', 'RAISE6SECRRP', 'RAISE6SECROP', 'RAISE6SECAPCFLAG', 'RAISE60SECRRP', 'RAISE60SECROP', 'RAISE60SECAPCFLAG', 'RAISE5MINRRP', 'RAISE5MINROP', 'RAISE5MINAPCFLAG', 'RAISEREGRRP', 'RAISEREGROP', 'RAISEREGAPCFLAG', 'LOWER6SECRRP', 'LOWER6SECROP', 'LOWER6SECAPCFLAG', 'LOWER60SECRRP', 'LOWER60SECROP', 'LOWER60SECAPCFLAG', 'LOWER5MINRRP', 'LOWER5MINROP', 'LOWER5MINAPCFLAG', 'LOWERREGRRP', 'LOWERREGROP', 'LOWERREGAPCFLAG', 'PRICE_STATUS', 'PRE_AP_ENERGY_PRICE', 'PRE_AP_RAISE6_PRICE', 'PRE_AP_RAISE60_PRICE', 'PRE_AP_RAISE5MIN_PRICE', 'PRE_AP_RAISEREG_PRICE', 'PRE_AP_LOWER6_PRICE', 'PRE_AP_LOWER60_PRICE', 'PRE_AP_LOWER5MIN_PRICE', 'PRE_AP_LOWERREG_PRICE', 'RAISE1SECRRP', 'RAISE1SECROP', 'RAISE1SECAPCFLAG', 'LOWER1SECRRP', 'LOWER1SECROP', 'LOWER1SECAPCFLAG', 'PRE_AP_RAISE1_PRICE', 'PRE_AP_LOWER1_PRICE', 'CUMUL_PRE_AP_ENERGY_

In [6]:
price_df = pd.DataFrame(price_table[1:], columns=price_table[0]).iloc[:, 4:]

In [7]:
price_df = price_df[["SETTLEMENTDATE", "REGIONID", "RUNNO", "INTERVENTION", "RRP", "PRICE_STATUS", "LASTCHANGED"]]
price_df

,SETTLEMENTDATE,REGIONID,RUNNO,INTERVENTION,RRP,PRICE_STATUS,LASTCHANGED
0,2026/11/15 05:15:00,NSW1,1,0,122.57447,FIRM,2026/11/15 05:10:01
1,2026/11/15 05:15:00,QLD1,1,0,115.6222,FIRM,2026/11/15 05:10:01
2,2026/11/15 05:15:00,SA1,1,0,120.44,FIRM,2026/11/15 05:10:01
3,2026/11/15 05:15:00,TAS1,1,0,110.08,FIRM,2026/11/15 05:10:01
4,2026/11/15 05:15:00,VIC1,1,0,116.83635,FIRM,2026/11/15 05:10:01


In [8]:
regionsum_table = []

with open("../data/tmp/PUBLIC_DISPATCHIS_202611150515_0000000548649038.CSV", "r") as f:
    for row in csv.reader(f):
        if row[1] == "DISPATCH" and row[2] == "REGIONSUM":
            regionsum_table.append(row)

print(regionsum_table)

[['I', 'DISPATCH', 'REGIONSUM', '9', 'SETTLEMENTDATE', 'RUNNO', 'REGIONID', 'DISPATCHINTERVAL', 'INTERVENTION', 'TOTALDEMAND', 'AVAILABLEGENERATION', 'AVAILABLELOAD', 'DEMANDFORECAST', 'DISPATCHABLEGENERATION', 'DISPATCHABLELOAD', 'NETINTERCHANGE', 'EXCESSGENERATION', 'LOWER5MINDISPATCH', 'LOWER5MINIMPORT', 'LOWER5MINLOCALDISPATCH', 'LOWER5MINLOCALPRICE', 'LOWER5MINLOCALREQ', 'LOWER5MINPRICE', 'LOWER5MINREQ', 'LOWER5MINSUPPLYPRICE', 'LOWER60SECDISPATCH', 'LOWER60SECIMPORT', 'LOWER60SECLOCALDISPATCH', 'LOWER60SECLOCALPRICE', 'LOWER60SECLOCALREQ', 'LOWER60SECPRICE', 'LOWER60SECREQ', 'LOWER60SECSUPPLYPRICE', 'LOWER6SECDISPATCH', 'LOWER6SECIMPORT', 'LOWER6SECLOCALDISPATCH', 'LOWER6SECLOCALPRICE', 'LOWER6SECLOCALREQ', 'LOWER6SECPRICE', 'LOWER6SECREQ', 'LOWER6SECSUPPLYPRICE', 'RAISE5MINDISPATCH', 'RAISE5MINIMPORT', 'RAISE5MINLOCALDISPATCH', 'RAISE5MINLOCALPRICE', 'RAISE5MINLOCALREQ', 'RAISE5MINPRICE', 'RAISE5MINREQ', 'RAISE5MINSUPPLYPRICE', 'RAISE60SECDISPATCH', 'RAISE60SECIMPORT', 'RAISE60S

In [9]:
regionsum_df = pd.DataFrame(regionsum_table[1:], columns=regionsum_table[0]).iloc[:, 4:]

In [11]:
regionsum_df = regionsum_df[["SETTLEMENTDATE", "REGIONID", "RUNNO", "INTERVENTION", "TOTALDEMAND"]]
regionsum_df

,SETTLEMENTDATE,REGIONID,RUNNO,INTERVENTION,TOTALDEMAND
0,2026/11/15 05:15:00,NSW1,1,0,6350.44
1,2026/11/15 05:15:00,QLD1,1,0,5786.74
2,2026/11/15 05:15:00,SA1,1,0,1117.01
3,2026/11/15 05:15:00,TAS1,1,0,924.62
4,2026/11/15 05:15:00,VIC1,1,0,4394.72
